# 01 — Exploratory Data Analysis
**Project:** Farmer Loan Default Predictor  
**Author:** Rajanithi N · AI & Data Science · DSU-SET

This notebook explores the farmer loan dataset to understand distributions, class balance, correlations, and feature relationships before model training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('../data/loan_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('Data Types:\n', df.dtypes)
print('\nMissing Values:\n', df.isnull().sum())
df.describe()

## 2. Target Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['default'].value_counts()
axes[0].bar(['No Default', 'Default'], counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Loan Default Distribution')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=['No Default', 'Default'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Default Rate')

plt.tight_layout()
plt.show()

print(f'Default rate: {df["default"].mean()*100:.1f}%')

## 3. Numerical Feature Distributions

In [ ]:
num_cols = ['age', 'land_area_acres', 'annual_income', 'loan_amount',
            'loan_tenure_months', 'previous_loans', 'credit_score']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue='default', kde=True, ax=axes[i],
                 palette={0: '#2ecc71', 1: '#e74c3c'}, alpha=0.6)
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
plt.suptitle('Numerical Feature Distributions by Default Status', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Categorical Feature Analysis

In [ ]:
cat_cols = ['crop_type', 'repayment_history', 'soil_quality', 'irrigation_type']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    default_rates = df.groupby(col)['default'].mean().sort_values(ascending=False)
    bars = axes[i].bar(default_rates.index, default_rates.values * 100,
                       color='#e67e22', edgecolor='black', alpha=0.85)
    axes[i].set_title(f'Default Rate by {col.replace("_", " ").title()}')
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, default_rates.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val*100:.0f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Default Rate by Categorical Features', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
encode_map = {
    'repayment_history': {'Good': 0, 'Poor': 1},
    'soil_quality': {'High': 0, 'Medium': 1, 'Low': 2},
    'irrigation_type': {'Canal': 0, 'Borewell': 1, 'Drip': 2, 'Rainfed': 3}
}
df_encoded = df.copy()
for col, mapping in encode_map.items():
    df_encoded[col] = df_encoded[col].map(mapping)

corr_cols = num_cols + ['repayment_history', 'soil_quality', 'irrigation_type', 'default']
corr = df_encoded[corr_cols].corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Key Observations

- **Credit score** and **repayment history** are the strongest predictors of default.
- Farmers with **rainfed irrigation** and **low soil quality** show higher default rates.
- Higher **previous loan count** correlates with increased default risk.
- **Annual income** and **land area** are inversely related to default probability.
- **Vegetables** and **Cotton** crops show higher default rates than **Rice** and **Wheat**.